# Atelier Préparation de Données Tabulaires

Contexte Une entreprise exploite plusieurs bâtiments intelligents équipés de capteurs IoT. Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la qualité de l'air, la consommation énergétique, le nombre de personnes présentes, le type de bâtiment, le mode de fonctionnement et l'état du système de climatisation. Les données collectées sont destinées à alimenter ultérieurement un modèle de Machine Learning capable de prédire la consommation énergétique ou de détecter les situations anormales. Cependant, les données brutes présentent volontairement différents problèmes : valeurs manquantes, doublons, valeurs aberrantes, types incorrects, valeurs incohérentes, variables catégorielles, catégories rares, déséquilibre des classes et échelles différentes entre variables. L'objectif de l'atelier est donc de transformer le fichier brut en un jeu de données propre et prêt pour le Machine Learning. 

## Partie 1 – Explorer les données 

### 1) Charger les données CSV ; 

In [1]:
import pandas as pd

df = pd.read_csv("../data/smart_building_raw.csv")

## 2) Afficher les premières lignes du dataset ;

In [2]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


## 3) Afficher les dernières lignes du dataset ; 

In [3]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


## 4) Combien d'observations contient le dataset ? 

In [4]:
print("Nombre d'observations :", df.shape[0])

Nombre d'observations : 507


## 5) Combien de variables possède le dataset ?

In [5]:
print("Nombre de variables :", df.shape[1])

Nombre de variables : 14


## 6) Identifier les variables numériques ; 

In [6]:
# df.select_dtypes(include='number') : selection tous les variables de types numerique
# .columns : recupere les noms de ces colonne sous forme de liste
variables_numeriques = df.select_dtypes(include='number').columns
print("Les variables numériques : ", list(variables_numeriques))

Les variables numériques :  ['id_mesure', 'temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']


## 7) Identifier les variables catégorielles ;

In [7]:
# df.select_dtypes(include='number') : selection tous les variables de types texte (objet)
variables_categorielles = df.select_dtypes(include='str').columns
print("Les variables catégorielles : ", list(variables_categorielles))

Les variables catégorielles :  ['date', 'batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


## 8) Identifier les dates

In [8]:
variables_dates = ['date']
print("variables de types dates : ", variables_dates)

variables de types dates :  ['date']


## 9) Identifier les identifiants

In [9]:
variables_identifiants = ['id_mesure']
print("variables de types identifiants : ", variables_identifiants)

variables de types identifiants :  ['id_mesure']


## 10) Déterminer les statistiques : moyenne, médiane, minimum, maximum, écart-type et quartiles.

In [10]:
df.describe()

,id_mesure,temperature,humidite,co2,occupation,consommation_kwh
count,507.000000,495.000000,496.000000,500.000000,501.000000,502.000000
mean,1251.114398,24.154141,57.864113,844.150000,44.850299,169.069323
std,144.782769,7.418465,16.026336,582.181386,24.949139,53.164294
min,1001.000000,-30.000000,-12.000000,89.000000,-20.000000,-100.000000
25%,1125.500000,21.600000,49.275000,623.750000,27.000000,136.875000
50%,1252.000000,24.000000,57.550000,787.500000,46.000000,169.800000
75%,1376.500000,26.000000,65.750000,952.000000,61.000000,202.975000
max,1500.000000,96.000000,160.000000,6000.000000,116.000000,336.200000


## 11) Y a-t-il des variables potentiellement problématiques ? 

D'après les statistiques descriptives, plusieurs variables présentent des valeurs incohérentes :

- **temperature** : valeurs négatives extrêmes (min = -30) et anormalement élevées (max = 96), physiquement impossibles pour un capteur intérieur
- **humidite** : valeurs négatives (min = -12) et supérieures à 100% (max = 160), impossibles pour une humidité relative
- **occupation** : valeur négative (min = -20), un nombre de personnes ne peut pas être négatif
- **consommation_kwh** : valeur négative (min = -100), une consommation ne peut pas être négative
- **co2** : valeur maximale très élevée (6000) comparée à la moyenne (~844), à examiner comme valeur extrême

De plus, les colonnes numériques ont des `count` différents (495 à 502 sur 507 lignes), ce qui indique la présence de **valeurs manquantes** en quantités variables selon la variable.

## 12) Pour les données incohérentes : 

## a) rechercher des valeurs telles que humidité < 0 ; 

In [11]:
humidite_negative = df[df['humidite'] < 0]
print("Nombre de valeurs d'humidité négatives :", len(humidite_negative))
humidite_negative[['id_mesure', 'humidite']]


Nombre de valeurs d'humidité négatives : 3


,id_mesure,humidite
281,1126,-5.0
335,1036,-8.0
366,1216,-12.0


## b) rechercher des valeurs telles que humidité > 100 ;

In [12]:
humidite_superieure_100 = df[df['humidite'] > 100]
print("Nombre de valeurs d'humidité > 100 :", len(humidite_superieure_100))
humidite_superieure_100[['id_mesure', 'humidite']]

Nombre de valeurs d'humidité > 100 : 7


,id_mesure,humidite
59,1246,108.0
103,1016,145.0
128,1156,160.0
160,1186,125.0
268,1276,140.0
327,1066,132.0
342,1096,110.0


## c) rechercher des valeurs telles que température extrêmement élevée ; 

In [13]:
# Seuil arbitraire mais raisonnable pour un capteur intérieur : au-delà de 50°C, la valeur est clairement aberrante
temperature_elevee = df[df['temperature'] > 50]
print("Nombre de valeurs de température extrêmement élevées :", len(temperature_elevee))
temperature_elevee[['id_mesure', 'temperature']]

Nombre de valeurs de température extrêmement élevées : 4


,id_mesure,temperature
7,1141,72.5
116,1181,96.0
161,1061,88.0
499,1021,95.2


## d) rechercher des valeurs telles que occupation négative

In [14]:
occupation_negative = df[df['occupation'] < 0]
print("Nombre de valeurs d'occupation negative :", len(occupation_negative))
occupation_negative[['id_mesure', 'occupation']]

Nombre de valeurs d'occupation negative : 5


,id_mesure,occupation
22,1031,-5.0
376,1231,-8.0
430,1081,-12.0
487,1131,-2.0
494,1331,-20.0


## e) rechercher des valeurs telles que consommation négative ; 

In [15]:
consommation_negative = df[df['consommation_kwh'] < 0]
print("Nombre de valeurs d'consommation negative :", len(consommation_negative))
consommation_negative[['id_mesure', 'consommation_kwh']]

Nombre de valeurs d'consommation negative : 4


,id_mesure,consommation_kwh
59,1246,-15.0
154,1046,-50.0
199,1146,-20.0
441,1346,-100.0


## f) Si une valeur est manifestement erronée et qu’on ne peut pas retrouver sa vraie valeur, la transformer en valeur manquante

In [16]:
# .loc[condition, colonne] cible précisément les cellules concernées, sans toucher au reste de la ligne

df.loc[df['humidite'] < 0, 'humidite'] = None          # 12a
df.loc[df['humidite'] > 100, 'humidite'] = None         # 12b
df.loc[df['temperature'] > 50, 'temperature'] = None    # 12c
df.loc[df['occupation'] < 0, 'occupation'] = None        # 12d
df.loc[df['consommation_kwh'] < 0, 'consommation_kwh'] = None  # 12e

# Vérification : le nombre de valeurs manquantes a bien augmenté sur ces colonnes
df[['humidite', 'temperature', 'occupation', 'consommation_kwh']].isna().sum()

humidite            21
temperature         16
occupation          11
consommation_kwh     9
dtype: int64

## g) rechercher des valeurs telles que catégories mal orthographiées. 

In [17]:
# des doublons dus à des fautes de frappe, majuscules incohérentes, espaces en trop, etc.
for col in ['batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']:
    print(col, ":", df[col].unique())
    print()

batiment : <StringArray>
['B8', 'B7', 'B5', 'B6', 'B3', 'B4', 'B2', 'B1']
Length: 8, dtype: str

type_batiment : <StringArray>
[         'Entrepôt',            'Bureau', 'Centre commercial',
        'Université',           'Hôpital',             'École',
             'ÉCOLE',             'ecole',            'BUREAU',
                 nan,             'Bureu',            'bureau',
       ' UNIVERSITÉ',          'hôpital ', 'centre commercial',
          ' Bureau ',          'entrepot']
Length: 17, dtype: str

zone : <StringArray>
['A', 'D', 'B', 'C']
Length: 4, dtype: str

mode_climatisation : <StringArray>
['Eco', 'Normal', 'Boost', nan, 'normal', 'BOOST', 'normale', 'Normal ']
Length: 8, dtype: str

etat_systeme : <StringArray>
['Normal', 'Alerte', 'Panne']
Length: 3, dtype: str

jour_semaine : <StringArray>
['Jeudi', 'Lundi', 'Dimanche', 'Vendredi', 'Mercredi', 'Mardi', 'Samedi', nan]
Length: 8, dtype: str

alerte : <StringArray>
['Non', 'Oui']
Length: 2, dtype: str



## h) normaliser les catégories textuelles en supprimant les espaces puis en uniformisant la casse

In [18]:
# .str.strip() supprime les espaces en début/fin de chaîne (ex: ' Bureau ' → 'Bureau')
# .str.lower() met tout en minuscules, pour uniformiser la casse (ex: 'BUREAU', 'bureau', 'Bureau' → 'bureau')
# .str.capitalize() remet ensuite une majuscule en début de mot (ex: 'bureau' → 'Bureau')

for col in ['type_batiment', 'mode_climatisation']:
    df[col] = df[col].str.strip().str.lower().str.capitalize()

# Vérification : on doit voir moins de valeurs uniques qu'avant
print("type_batiment :", df['type_batiment'].unique())
print()
print("mode_climatisation :", df['mode_climatisation'].unique())

type_batiment : <StringArray>
[         'Entrepôt',            'Bureau', 'Centre commercial',
        'Université',           'Hôpital',             'École',
             'Ecole',                 nan,             'Bureu',
          'Entrepot']
Length: 10, dtype: str

mode_climatisation : <StringArray>
['Eco', 'Normal', 'Boost', nan, 'Normale']
Length: 5, dtype: str


In [19]:
# .replace() remplace des valeurs précises par la bonne version, en se basant sur un dictionnaire {ancienne: nouvelle}
df['type_batiment'] = df['type_batiment'].replace({
    'Ecole': 'École',
    'Entrepot': 'Entrepôt',
    'Bureu': 'Bureau'
})

df['mode_climatisation'] = df['mode_climatisation'].replace({
    'Normale': 'Normal'
})

# Vérification finale
print("type_batiment :", df['type_batiment'].unique())
print()
print("mode_climatisation :", df['mode_climatisation'].unique())

type_batiment : <StringArray>
[         'Entrepôt',            'Bureau', 'Centre commercial',
        'Université',           'Hôpital',             'École',
                 nan]
Length: 7, dtype: str

mode_climatisation : <StringArray>
['Eco', 'Normal', 'Boost', nan]
Length: 4, dtype: str


## 13) Pour les valeurs manquantes : 

## a) Calculer le nombre et le pourcentage de valeurs manquantes par colonne 

In [20]:
# .isna().sum() compte le nombre de valeurs manquantes (NaN) pour chaque colonne
nb_manquantes = df.isna().sum()

# On calcule le pourcentage en divisant par le nombre total de lignes, puis en multipliant par 100
pct_manquantes = (nb_manquantes / len(df)) * 100

# On assemble les deux résultats dans un seul DataFrame pour une lecture plus claire
resume_manquantes = pd.DataFrame({
    'nb_manquantes': nb_manquantes,
    'pct_manquantes': pct_manquantes.round(2)
})
resume_manquantes

,nb_manquantes,pct_manquantes
id_mesure,0,0.00
date,0,0.00
batiment,0,0.00
type_batiment,4,0.79
zone,0,0.00
temperature,16,3.16
humidite,21,4.14
co2,7,1.38
occupation,11,2.17
consommation_kwh,9,1.78


## b) Quelle variable possède le plus de valeurs manquantes ? 

C'est **humidite** qui a le plus de valeurs manquantes : 21 valeurs manquantes, soit 4.14% des observations.

## c) Quelle stratégie utiliser pour les valeurs manquantes ? 

Aucune colonne n'a plus de 5% de valeurs manquantes ici (le maximum est 4.14% pour humidite) les pourcentages sont donc faibles, ce qui permet d'**imputer** (remplacer) les valeurs manquantes plutôt que de supprimer des lignes entières :

- Pour les **variables numériques** (temperature, humidite, co2, occupation, consommation_kwh) : imputation par la **médiane**, plus robuste que la moyenne face aux valeurs extrêmes encore présentes dans les données.
- Pour les **variables catégorielles** (type_batiment, mode_climatisation, jour_semaine) : imputation par le **mode** (la valeur la plus fréquente).

On évite de supprimer les lignes, car même avec un faible taux de valeurs manquantes par colonne, additionner les colonnes concernées ferait perdre une part non négligeable des 507 observations disponibles.

## d) Peut-on supprimer toutes les lignes contenant des valeurs manquantes ?

Techniquement oui (`df.dropna()`), mais ce n'est **pas recommandé** ici, pour plusieurs raisons :

- Plusieurs colonnes ont des valeurs manquantes différentes selon les lignes (temperature, humidite, co2, occupation, consommation_kwh, type_batiment, mode_climatisation, jour_semaine) en cumulant, le nombre de lignes concernées par au moins une valeur manquante serait supérieur au pourcentage max d'une seule colonne (4.14%).
- Supprimer des lignes fait perdre de l'information utile sur les autres variables de cette même ligne (si seule l'humidité manque, la température, le CO2, etc. de cette ligne restent valides et exploitables).
- Sur un jeu de données de seulement 507 observations, chaque ligne supprimée réduit sensiblement la taille du dataset disponible pour l'entraînement du modèle.

L'imputation (point 13c) est donc préférable ici à la suppression.

## e) Dans quels cas utiliser la moyenne ? 

La moyenne est adaptée quand la variable numérique suit une **distribution symétrique**, sans valeurs extrêmes (outliers) qui la déséquilibrent.

Dans ce cas, la moyenne reflète bien la "valeur typique" de la variable. Mais si la distribution est asymétrique ou contient des valeurs aberrantes, la moyenne est tirée vers ces extrêmes et devient moins représentative c'est là que la médiane est préférable.

Ici, avant le nettoyage des points 12a-12e, plusieurs variables (temperature, humidite, occupation, consommation_kwh) avaient des valeurs extrêmes qui auraient faussé la moyenne ce qui justifie le choix de la médiane fait au point 13c, plutôt que la moyenne.

## f) Quand préférer la médiane ? 

La médiane est préférable quand la variable a une **distribution asymétrique** ou contient des **valeurs extrêmes** (outliers)contrairement à la moyenne, la médiane n'est pas influencée par des valeurs très éloignées du reste des données, puisqu'elle se base uniquement sur la position centrale des valeurs triées, pas sur leur amplitude.

C'est le cas ici pour les variables numériques du dataset (temperature, humidite, co2, occupation, consommation_kwh) : même après correction des incohérences les plus flagrantes (points 12a-12e), il reste des valeurs extrêmes légitimes (par exemple co2 = 6000, largement au-dessus de la moyenne ~844) qui pourraient encore tirer la moyenne vers le haut. La médiane reste donc le choix le plus sûr pour ces variables.

## g) Comment traiter une variable catégorielle ? 

Pour une variable catégorielle (type_batiment, mode_climatisation, jour_semaine), la moyenne et la médiane n'ont pas de sens (ce ne sont pas des nombres) on utilise plutôt :

- **Le mode** (la valeur la plus fréquente) : c'est la méthode la plus courante et la plus simple, adaptée quand une catégorie domine nettement les autres.
- **Une catégorie dédiée** (par exemple "Inconnu" ou "Manquant") : utile quand on veut éviter de biaiser artificiellement vers la catégorie majoritaire, ou quand l'absence de valeur pourrait elle-même être informative.

Ici, avec de faibles taux de valeurs manquantes (0.79% à 0.99% selon la colonne), l'imputation par le **mode** est suffisante et cohérente avec la stratégie retenue au point 13c.